<a href="https://colab.research.google.com/github/jarekwan/PROJEKT_SCANNER/blob/main/2Model_danych.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
os.makedirs('/content/drive/MyDrive/projekt_test', exist_ok=True)

print("folder ready")

In [ ]:
%%writefile /content/drive/MyDrive/projekt_test/modul_model_danych.py

from __future__ import annotations

import json
from dataclasses import dataclass, field, fields
from datetime import datetime
from pathlib import Path
from statistics import mean, pstdev
from typing import Any

import pandas as pd


FOLDER_PROJEKTU = Path("/content/drive/MyDrive/projekt_test")


@dataclass(
    frozen=True,
    slots=True,
    order=True
)
class Notowanie:
    data: datetime

    open: float = field(compare=False)
    high: float = field(compare=False)
    low: float = field(compare=False)
    close: float = field(compare=False)
    volume: int = field(compare=False)

    def __post_init__(self) -> None:
        if self.open < 0:
            raise ValueError("open nie moze byc ujemne")

        if self.high < 0:
            raise ValueError("high nie moze byc ujemne")

        if self.low < 0:
            raise ValueError("low nie moze byc ujemne")

        if self.close < 0:
            raise ValueError("close nie moze byc ujemne")

        if self.volume < 0:
            raise ValueError("volume nie moze byc ujemne")

        if self.high < self.low:
            raise ValueError(
                "high nie moze byc mniejsze od low"
            )

        if not self.low <= self.open <= self.high:
            raise ValueError(
                "open musi znajdowac sie pomiedzy low i high"
            )

        if not self.low <= self.close <= self.high:
            raise ValueError(
                "close musi znajdowac sie pomiedzy low i high"
            )


@dataclass(
    slots=True,
    kw_only=True
)
class Spolka:
    ticker: str
    nazwa: str = ""
    sektor: str | None = None
    branza: str | None = None
    kraj: str | None = None

    tagi: list[str] = field(
        default_factory=list
    )

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        repr=False
    )

    def __post_init__(self) -> None:
        self.ticker = self.ticker.strip().upper()

        if not self.ticker:
            raise ValueError(
                "ticker nie moze byc pusty"
            )

        self.nazwa = self.nazwa.strip()

        if self.sektor is not None:
            self.sektor = self.sektor.strip()

        if self.branza is not None:
            self.branza = self.branza.strip()

        if self.kraj is not None:
            self.kraj = self.kraj.strip()

        self.tagi = [
            tag.strip().lower()
            for tag in self.tagi
            if tag.strip()
        ]


@dataclass(
    slots=True,
    kw_only=True
)
class SpolkaDywidendowa(Spolka):
    dywidenda_roczna: float = 0.0
    stopa_dywidendy: float = 0.0
    liczba_lat_dywidendy: int = 0

    def __post_init__(self) -> None:
        Spolka.__post_init__(self)

        if self.dywidenda_roczna < 0:
            raise ValueError(
                "dywidenda roczna nie moze byc ujemna"
            )

        if self.stopa_dywidendy < 0:
            raise ValueError(
                "stopa dywidendy nie moze byc ujemna"
            )

        if self.liczba_lat_dywidendy < 0:
            raise ValueError(
                "liczba lat dywidendy nie moze byc ujemna"
            )


@dataclass(
    slots=True,
    kw_only=True
)
class HistoriaCen:
    spolka: Spolka

    notowania: list[Notowanie] = field(
        default_factory=list
    )

    liczba_notowan: int = field(
        init=False
    )

    srednia_cena: float = field(
        init=False
    )

    momentum: float = field(
        init=False
    )

    zmiennosc: float = field(
        init=False
    )

    def __post_init__(self) -> None:
        if not self.notowania:
            raise ValueError(
                "historia cen nie moze byc pusta"
            )

        self.notowania.sort(
            key=lambda n: n.data
        )

        self.liczba_notowan = len(
            self.notowania
        )

        self.srednia_cena = (
            self._oblicz_srednia()
        )

        self.momentum = (
            self._oblicz_momentum()
        )

        self.zmiennosc = (
            self._oblicz_zmiennosc()
        )

    def _oblicz_srednia(self) -> float:
        ceny = [
            n.close
            for n in self.notowania
        ]

        return mean(ceny)

    def _oblicz_momentum(self) -> float:
        if len(self.notowania) < 2:
            return 0.0

        pierwsza = self.notowania[0].close
        ostatnia = self.notowania[-1].close

        if pierwsza == 0:
            return 0.0

        return (
            (ostatnia - pierwsza)
            / pierwsza
        ) * 100.0

    def _oblicz_zmiennosc(self) -> float:
        if len(self.notowania) < 2:
            return 0.0

        zwroty: list[float] = []

        for i in range(
            1,
            len(self.notowania)
        ):
            poprzednia = (
                self.notowania[i - 1].close
            )

            aktualna = (
                self.notowania[i].close
            )

            if poprzednia == 0:
                continue

            zwrot = (
                aktualna - poprzednia
            ) / poprzednia

            zwroty.append(zwrot)

        if len(zwroty) < 2:
            return 0.0

        return (
            pstdev(zwroty)
            * 100.0
        )

    @property
    def ticker(self) -> str:
        return self.spolka.ticker

    @property
    def ostatnie_notowanie(
        self
    ) -> Notowanie:
        return self.notowania[-1]

    @property
    def ostatnia_cena(
        self
    ) -> float:
        return self.ostatnie_notowanie.close


@dataclass(
    frozen=True,
    slots=True,
    kw_only=True
)
class WynikSkanowania:
    spolka: Spolka = field(
        compare=True
    )

    historia: HistoriaCen = field(
        compare=False
    )

    sygnal: str = field(
        default="NEUTRAL",
        compare=True
    )

    wynik_punktowy: float = field(
        init=False,
        compare=True
    )

    srednia: float = field(
        init=False,
        compare=False
    )

    momentum: float = field(
        init=False,
        compare=False
    )

    zmiennosc: float = field(
        init=False,
        compare=False
    )

    timestamp: datetime = field(
        default_factory=datetime.now,
        compare=False,
        repr=False
    )

    dodatkowe_dane: dict[str, Any] = field(
        default_factory=dict,
        compare=False
    )

    def __post_init__(self) -> None:
        if (
            self.spolka.ticker
            != self.historia.ticker
        ):
            raise ValueError(
                "spolka i historia dotycza "
                "roznych tickerow"
            )

        object.__setattr__(
            self,
            "srednia",
            self.historia.srednia_cena
        )

        object.__setattr__(
            self,
            "momentum",
            self.historia.momentum
        )

        object.__setattr__(
            self,
            "zmiennosc",
            self.historia.zmiennosc
        )

        object.__setattr__(
            self,
            "wynik_punktowy",
            self._oblicz_wynik()
        )

    def _oblicz_wynik(self) -> float:
        punkty = 0.0

        if self.momentum > 0:
            punkty += min(
                self.momentum,
                50.0
            )

        elif self.momentum < 0:
            punkty += max(
                self.momentum,
                -50.0
            )

        if self.zmiennosc < 2.0:
            punkty += 10.0

        elif self.zmiennosc > 5.0:
            punkty -= 10.0

        return round(
            punkty,
            2
        )


def parse_float(
    wartosc: Any,
    nazwa: str
) -> float:

    try:
        return float(wartosc)

    except (TypeError, ValueError) as e:
        raise ValueError(
            f"niepoprawna wartosc {nazwa}: "
            f"{wartosc}"
        ) from e


def parse_int(
    wartosc: Any,
    nazwa: str
) -> int:

    try:
        return int(float(wartosc))

    except (TypeError, ValueError) as e:
        raise ValueError(
            f"niepoprawna wartosc {nazwa}: "
            f"{wartosc}"
        ) from e


def parse_date(
    wartosc: Any
) -> datetime:

    try:
        return datetime.strptime(
            str(wartosc),
            "%Y-%m-%d"
        )

    except ValueError as e:
        raise ValueError(
            f"niepoprawna data: {wartosc}"
        ) from e


def wczytaj_historie_z_modulu_1(
    ticker: str,
    folder: Path = FOLDER_PROJEKTU
) -> HistoriaCen:

    ticker = ticker.strip().upper()

    plik = (
        folder
        / f"{ticker}_twelve.csv"
    )

    if not plik.exists():
        raise FileNotFoundError(
            f"brak pliku z modulu 1: {plik}"
        )

    df = pd.read_csv(plik)

    wymagane = [
        "data",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]

    brakujace = [
        kolumna
        for kolumna in wymagane
        if kolumna not in df.columns
    ]

    if brakujace:
        raise ValueError(
            "brak wymaganych kolumn: "
            f"{brakujace}"
        )

    notowania: list[Notowanie] = []

    for _, row in df.iterrows():
        notowanie = Notowanie(
            data=parse_date(
                row["data"]
            ),
            open=parse_float(
                row["open"],
                "open"
            ),
            high=parse_float(
                row["high"],
                "high"
            ),
            low=parse_float(
                row["low"],
                "low"
            ),
            close=parse_float(
                row["close"],
                "close"
            ),
            volume=parse_int(
                row["volume"],
                "volume"
            )
        )

        notowania.append(
            notowanie
        )

    spolka = Spolka(
        ticker=ticker
    )

    return HistoriaCen(
        spolka=spolka,
        notowania=notowania
    )


def historia_do_dict(
    historia: HistoriaCen
) -> dict[str, Any]:

    return {
        "spolka": {
            "ticker": historia.spolka.ticker,
            "nazwa": historia.spolka.nazwa,
            "sektor": historia.spolka.sektor,
            "branza": historia.spolka.branza,
            "kraj": historia.spolka.kraj,
            "tagi": historia.spolka.tagi
        },

        "statystyki": {
            "liczba_notowan":
                historia.liczba_notowan,

            "srednia_cena":
                historia.srednia_cena,

            "momentum":
                historia.momentum,

            "zmiennosc":
                historia.zmiennosc,

            "ostatnia_cena":
                historia.ostatnia_cena
        },

        "notowania": [
            {
                "data":
                    n.data.strftime(
                        "%Y-%m-%d"
                    ),

                "open": n.open,
                "high": n.high,
                "low": n.low,
                "close": n.close,
                "volume": n.volume
            }

            for n in historia.notowania
        ]
    }


def zapisz_model(
    historia: HistoriaCen,
    folder: Path = FOLDER_PROJEKTU
) -> Path:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

    plik = (
        folder
        / f"{historia.ticker}_model.json"
    )

    dane = historia_do_dict(
        historia
    )

    with open(
        plik,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            dane,
            f,
            ensure_ascii=False,
            indent=2
        )

    return plik


def pokaz_dataclass(
    obj: Any
) -> None:

    print(
        f"\nDATACLASS: "
        f"{type(obj).__name__}"
    )

    for f in fields(obj):
        print(
            f"{f.name:20} "
            f"typ={f.type} "
            f"init={f.init} "
            f"compare={f.compare}"
        )


def run() -> None:

    ticker = input(
        "podaj ticker: "
    ).strip().upper()

    print(
        "\nwczytywanie danych "
        "zapisanych przez modul 1..."
    )

    historia = (
        wczytaj_historie_z_modulu_1(
            ticker
        )
    )

    print(
        "wczytano:",
        historia.liczba_notowan,
        "notowan"
    )

    print(
        "ticker:",
        historia.ticker
    )

    print(
        "srednia cena:",
        round(
            historia.srednia_cena,
            2
        )
    )

    print(
        "momentum:",
        round(
            historia.momentum,
            2
        ),
        "%"
    )

    print(
        "zmiennosc:",
        round(
            historia.zmiennosc,
            2
        ),
        "%"
    )

    print(
        "ostatnia cena:",
        historia.ostatnia_cena
    )

    plik = zapisz_model(
        historia
    )

    print(
        "\nzapisano model:"
    )

    print(
        plik
    )

    pokaz_dataclass(
        historia.spolka
    )

    pokaz_dataclass(
        historia.notowania[0]
    )

    pokaz_dataclass(
        historia
    )

    print(
        "\nMODUL MODEL DANYCH "
        "DZIALA POPRAWNIE"
    )